# Đánh giá lại experiment SASRec trên tập test

Notebook nạp checkpoint tốt nhất, tái tạo tập test, tính metric và hiển thị một số mẫu mà item đúng không nằm trong Top-K.

## 1. Thiết lập đường dẫn và import thư viện

In [1]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Không tìm thấy project root từ {cwd}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.evaluation_dataset import EvaluationDataset
from src.models.sasrec import SASRec
from src.utils.io import load_pickle

experiment_dir = project_root / "outputs/sasrec_baseline/2026-08-14_09-07-35"
processed_dir = project_root / "data/processed/mindsmall_v1"
checkpoint_path = experiment_dir / "checkpoints/best_model.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for path in (experiment_dir / "config.yaml", experiment_dir / "run_info.json", checkpoint_path):
    if not path.is_file():
        raise FileNotFoundError(path)

print(f"Project root: {project_root}")
print(f"Experiment: {experiment_dir}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Device: {device}")

Project root: /home/trieu/Intern/SwM_precomputed
Experiment: /home/trieu/Intern/SwM_precomputed/outputs/sasrec_baseline/2026-08-14_09-07-35
Checkpoint: /home/trieu/Intern/SwM_precomputed/outputs/sasrec_baseline/2026-08-14_09-07-35/checkpoints/best_model.pt
Device: cuda


## 2–5. Đọc experiment, dựng tập test và nạp checkpoint

In [2]:
with (experiment_dir / "config.yaml").open(encoding="utf-8") as file:
    config = yaml.safe_load(file)
with (experiment_dir / "run_info.json").open(encoding="utf-8") as file:
    run_info = json.load(file)

seed = int(config["training"]["seed"])
max_sequence_length = int(config["data"]["max_sequence_length"])
embedding_dim = int(config["model"]["embedding_dim"])
num_negatives = int(config["evaluation"]["num_negatives"])
k = int(config["evaluation"]["k"])
batch_size = int(config["training"]["batch_size"])
num_workers = int(config["training"].get("num_workers", 0))

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

test_samples = load_pickle(processed_dir / "artifacts/test_samples.pkl")
news_vector_mapping = load_pickle(processed_dir / "artifacts/news_vector_mapping.pkl")
test_dataset = EvaluationDataset(
    samples=test_samples,
    num_negatives=num_negatives,
    max_sequence_length=max_sequence_length,
    padding_id=int(config["data"]["padding_id"]),
    mapping=news_vector_mapping,
    vector_size=embedding_dim,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=device.type == "cuda",
    persistent_workers=num_workers > 0,
)

model = SASRec(
    max_sequence_length=max_sequence_length,
    embedding_dim=embedding_dim,
    num_blocks=int(config["model"]["num_blocks"]),
    num_heads=int(config["model"]["num_heads"]),
    dropout=float(config["model"]["dropout"]),
)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
load_result = model.load_state_dict(state_dict, strict=False)
if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError(f"Missing={load_result.missing_keys}; Unexpected={load_result.unexpected_keys}")
model = model.to(device).eval()

pd.Series({
    "seed": seed,
    "test_samples": len(test_dataset),
    "mapped_news": len(news_vector_mapping),
    "candidates_per_sample": num_negatives + 1,
    "batch_size": batch_size,
    "max_sequence_length": max_sequence_length,
    "embedding_dim": embedding_dim,
    "num_blocks": config["model"]["num_blocks"],
    "num_heads": config["model"]["num_heads"],
    "checkpoint_epoch": checkpoint.get("epoch"),
    "saved_test_metrics": run_info.get("test_metrics"),
})

seed                                                                    42
test_samples                                                         53755
mapped_news                                                          65238
candidates_per_sample                                                  101
batch_size                                                            1024
max_sequence_length                                                     50
embedding_dim                                                          384
num_blocks                                                               4
num_heads                                                                1
checkpoint_epoch                                                        24
saved_test_metrics       {'hit_rate@10': 0.37868105285545356, 'ndcg@10'...
dtype: object

## 6. Chạy suy luận

Item đúng luôn ở ứng viên 0. Negative trong `EvaluationDataset` đã loại các item thuộc history; seed được đặt trước khi tạo dataset để lấy mẫu tái lập.

In [3]:
rows = []
offset = 0
with torch.no_grad():
    for batch in test_loader:
        inputs = batch["input_vectors"].to(device, non_blocking=True)
        candidates = batch["candidate_vectors"].to(device, non_blocking=True)
        scores = model.score_candidates(inputs, candidates).cpu()
        ranks = (scores > scores[:, :1]).sum(dim=1) + 1
        sorted_indices = scores.argsort(dim=1, descending=True)

        for local_index in range(scores.size(0)):
            sample_index = offset + local_index
            source = test_samples[sample_index]
            evaluated = test_dataset.evaluation_samples[sample_index]
            candidate_ids = [evaluated["target"], *evaluated["negatives"]]
            order = sorted_indices[local_index].tolist()
            best_negative_index = next(index for index in order if index != 0)
            top_indices = order[:k]
            rows.append({
                "sample_index": sample_index,
                "history_length": len(source["history"]),
                "history": source["history"],
                "target_id": source["target"],
                "positive_rank": int(ranks[local_index]),
                "positive_score": float(scores[local_index, 0]),
                "best_negative_id": candidate_ids[best_negative_index],
                "best_negative_score": float(scores[local_index, best_negative_index]),
                "score_margin": float(scores[local_index, 0] - scores[local_index, best_negative_index]),
                "top_k_ids": [candidate_ids[index] for index in top_indices],
                "top_k_scores": [float(scores[local_index, index]) for index in top_indices],
            })
        offset += scores.size(0)

results_df = pd.DataFrame(rows)
results_df.head()

,sample_index,history_length,history,target_id,positive_rank,positive_score,best_negative_id,best_negative_score,score_margin,top_k_ids,top_k_scores
0,0,44,"[N41777, N10629, N27448, N30665, N15077, N4148...",N55237,9,1.273219,N4303,3.053585,-1.780365,"[N4303, N2646, N6196, N6367, N45971, N63993, N...","[3.053584575653076, 2.8994975090026855, 2.6280..."
1,1,14,"[N33998, N47765, N56742, N36511, N44796, N2214...",N53572,15,0.509371,N56337,3.123507,-2.614136,"[N56337, N39186, N14536, N31799, N1259, N10449...","[3.123507261276245, 2.3401222229003906, 1.9569..."
2,2,22,"[N44251, N27612, N36699, N40467, N26015, N6199...",N46162,36,-0.663559,N4296,3.456089,-4.119648,"[N4296, N60830, N20841, N35543, N23874, N21666...","[3.4560887813568115, 3.046032667160034, 2.4092..."
3,3,6,"[N33276, N2186, N250, N34323, N15476, N36424]",N62365,6,1.750307,N58498,2.158676,-0.408370,"[N58498, N35735, N44283, N45255, N620, N62365,...","[2.1586763858795166, 2.129572629928589, 2.0252..."
4,4,28,"[N24073, N6974, N29911, N63019, N5391, N54496,...",N56969,2,3.166877,N16536,3.173571,-0.006694,"[N16536, N56969, N12075, N17840, N27264, N1456...","[3.1735706329345703, 3.166876792907715, 2.7399..."


## 7–8. Tính metric và thu thập trường hợp sai

In [4]:
metrics = {}
ranks = results_df["positive_rank"]
for cutoff in sorted({1, 5, 10, k}):
    hits = ranks <= cutoff
    metrics[f"hit_rate@{cutoff}"] = hits.mean()
    metrics[f"recall@{cutoff}"] = hits.mean()  # Một item relevant/mẫu.
    metrics[f"ndcg@{cutoff}"] = np.where(hits, 1.0 / np.log2(ranks + 1), 0.0).mean()
metrics["mrr"] = (1.0 / ranks).mean()

metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["recomputed"])
metrics_df["saved_in_run_info"] = pd.Series(run_info.get("test_metrics", {}))
metrics_df["difference"] = metrics_df["recomputed"] - metrics_df["saved_in_run_info"]
display(metrics_df)
display(ranks.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("positive_rank"))

results_df["hit_at_k"] = ranks <= k
results_df["ndcg_at_k"] = np.where(results_df["hit_at_k"], 1.0 / np.log2(ranks + 1), 0.0)
errors_df = (
    results_df.loc[~results_df["hit_at_k"]]
    .sort_values(["positive_rank", "score_margin"], ascending=[True, True])
    .reset_index(drop=True)
)
print(f"Sai tại K={k}: {len(errors_df):,}/{len(results_df):,} ({len(errors_df) / len(results_df):.2%})")
errors_df[["sample_index", "target_id", "positive_rank", "positive_score", "best_negative_id", "best_negative_score", "score_margin"]].head(10)

,recomputed,saved_in_run_info,difference
hit_rate@1,0.062190,NaN,NaN
recall@1,0.062190,NaN,NaN
ndcg@1,0.062190,NaN,NaN
hit_rate@5,0.242433,NaN,NaN
recall@5,0.242433,NaN,NaN
ndcg@5,0.152592,NaN,NaN
hit_rate@10,0.379146,0.378681,0.000465
recall@10,0.379146,NaN,NaN
ndcg@10,0.196498,0.196174,0.000323
mrr,0.164245,NaN,NaN


,positive_rank
count,53755.000000
mean,25.331764
std,23.475389
min,1.000000
50%,17.000000
75%,40.000000
90%,62.000000
95%,73.000000
99%,88.000000
max,101.000000


Sai tại K=10: 33,374/53,755 (62.09%)


,sample_index,target_id,positive_rank,positive_score,best_negative_id,best_negative_score,score_margin
0,29445,N55237,11,0.866188,N45145,5.931194,-5.065006
1,47066,N40956,11,0.055939,N35921,4.788934,-4.732996
2,28469,N29862,11,-0.041013,N28810,4.633442,-4.674456
3,14106,N58656,11,1.032198,N59267,5.648886,-4.616688
4,45367,N18774,11,1.051739,N33352,5.558236,-4.506497
5,9785,N26485,11,0.563536,N27004,5.054064,-4.490528
6,46947,N53863,11,0.833978,N54991,5.262932,-4.428954
7,9033,N3168,11,1.128016,N39067,5.445724,-4.317708
8,30156,N28682,11,0.655800,N59923,4.935228,-4.279428
9,12596,N47383,11,0.729781,N51396,5.006535,-4.276753


## 9. In một số mẫu đánh sai và tùy chọn xuất CSV

In [5]:
NUM_ERRORS_TO_SHOW = 20
RECENT_HISTORY_ITEMS = 20
SAVE_ERRORS_CSV = False

for display_index, row in errors_df.head(NUM_ERRORS_TO_SHOW).iterrows():
    print("=" * 100)
    print(f"Lỗi #{display_index + 1} | sample={row['sample_index']} | rank đúng={row['positive_rank']} | K={k}")
    print(f"History gần nhất : {row['history'][-RECENT_HISTORY_ITEMS:]}")
    print(f"Item đúng        : {row['target_id']} (score={row['positive_score']:.6f})")
    print(f"Negative tốt nhất: {row['best_negative_id']} (score={row['best_negative_score']:.6f})")
    print(f"Margin           : {row['score_margin']:.6f}")
    print("Top-K dự đoán:")
    for position, (item_id, score) in enumerate(zip(row["top_k_ids"], row["top_k_scores"]), start=1):
        print(f"  {position:>2}. {item_id:<10} score={score:.6f}")

if SAVE_ERRORS_CSV:
    output_path = experiment_dir / "predictions/test_error_examples.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    errors_df.to_csv(output_path, index=False)
    print(f"Đã lưu CSV: {output_path}")

Lỗi #1 | sample=29445 | rank đúng=11 | K=10
History gần nhất : ['N17973', 'N20397', 'N42362', 'N23958', 'N13168', 'N64248', 'N53901', 'N268', 'N39798', 'N40070', 'N4080', 'N21666', 'N37091', 'N11363', 'N9836', 'N7535', 'N3128', 'N62853', 'N56227', 'N4247']
Item đúng        : N55237 (score=0.866188)
Negative tốt nhất: N45145 (score=5.931194)
Margin           : -5.065006
Top-K dự đoán:
   1. N45145     score=5.931194
   2. N22351     score=4.351355
   3. N39461     score=3.759094
   4. N19079     score=3.620041
   5. N22039     score=3.169299
   6. N32006     score=2.901478
   7. N4147      score=2.624988
   8. N41123     score=1.058545
   9. N40987     score=0.988393
  10. N5697      score=0.900404
Lỗi #2 | sample=47066 | rank đúng=11 | K=10
History gần nhất : ['N33154', 'N64186', 'N28088', 'N37748', 'N19435', 'N42620', 'N60702', 'N4607', 'N61972', 'N55911', 'N46196', 'N15670', 'N1985', 'N2003', 'N33548', 'N53017', 'N28091', 'N35671', 'N7938', 'N64536']
Item đúng        : N40956 (score=

## 10. Phân tích nguyên nhân của các lỗi

Phần này kiểm tra các **yếu tố liên quan** đến lỗi: lịch sử ngắn/dài, target hiếm, thay đổi chủ đề, độ tương đồng nội dung, negative trùng và độ khó của bộ ứng viên. Đây là phân tích chẩn đoán; tương quan không tự nó chứng minh quan hệ nhân quả.

In [6]:
from src.data.parser import NEWS_COLUMNS

news_frames = []
for news_path in (
    project_root / "data/raw/MINDsmall_train/news.tsv",
    project_root / "data/raw/MINDsmall_dev/news.tsv",
):
    frame = pd.read_csv(news_path, sep="\t", names=NEWS_COLUMNS)
    news_frames.append(frame[["news_id", "category", "subcategory", "title"]])

news_metadata = (
    pd.concat(news_frames, ignore_index=True)
    .drop_duplicates("news_id", keep="last")
    .set_index("news_id")
)

train_samples = load_pickle(processed_dir / "artifacts/train_samples.pkl")
train_target_frequency = pd.Series(
    (sample["target"] for sample in train_samples), dtype="object"
).value_counts()

# Chuẩn hóa vector để cosine similarity không phụ thuộc độ lớn vector.
def normalized_news_vector(news_id):
    vector = np.asarray(news_vector_mapping[news_id], dtype=np.float32)
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 0 else vector

analysis_rows = []
for row in rows:
    sample_index = row["sample_index"]
    evaluated = test_dataset.evaluation_samples[sample_index]
    history_ids = [item for item in evaluated["history"] if item != config["data"]["padding_id"]]
    candidate_ids = [evaluated["target"], *evaluated["negatives"]]
    target_id = row["target_id"]

    target_vector = normalized_news_vector(target_id)
    history_vectors = np.stack([normalized_news_vector(item) for item in history_ids])
    negative_vectors = np.stack([normalized_news_vector(item) for item in evaluated["negatives"]])
    target_history_similarity = history_vectors @ target_vector
    target_negative_similarity = negative_vectors @ target_vector

    target_category = news_metadata.at[target_id, "category"] if target_id in news_metadata.index else None
    history_categories = [
        news_metadata.at[item, "category"]
        for item in history_ids
        if item in news_metadata.index
    ]
    dominant_category = pd.Series(history_categories).mode().iat[0] if history_categories else None

    analysis_rows.append({
        "sample_index": sample_index,
        "target_train_frequency": int(train_target_frequency.get(target_id, 0)),
        "target_unseen_in_train": target_id not in train_target_frequency.index,
        "target_category": target_category,
        "history_dominant_category": dominant_category,
        "category_shift": bool(target_category and dominant_category and target_category != dominant_category),
        "target_max_history_cosine": float(target_history_similarity.max()),
        "target_mean_history_cosine": float(target_history_similarity.mean()),
        "hardest_negative_target_cosine": float(target_negative_similarity.max()),
        "unique_candidate_ratio": len(set(candidate_ids)) / len(candidate_ids),
        "duplicate_negative_count": len(evaluated["negatives"]) - len(set(evaluated["negatives"])),
    })

analysis_df = results_df.merge(pd.DataFrame(analysis_rows), on="sample_index", validate="one_to_one")
analysis_df.head()

,sample_index,history_length,history,target_id,positive_rank,positive_score,best_negative_id,best_negative_score,score_margin,top_k_ids,...,target_train_frequency,target_unseen_in_train,target_category,history_dominant_category,category_shift,target_max_history_cosine,target_mean_history_cosine,hardest_negative_target_cosine,unique_candidate_ratio,duplicate_negative_count
0,0,44,"[N41777, N10629, N27448, N30665, N15077, N4148...",N55237,9,1.273219,N4303,3.053585,-1.780365,"[N4303, N2646, N6196, N6367, N45971, N63993, N...",...,0,True,movies,sports,True,0.388633,0.058087,0.321143,1.0,0
1,1,14,"[N33998, N47765, N56742, N36511, N44796, N2214...",N53572,15,0.509371,N56337,3.123507,-2.614136,"[N56337, N39186, N14536, N31799, N1259, N10449...",...,0,True,music,health,True,0.254609,0.063700,0.403184,1.0,0
2,2,22,"[N44251, N27612, N36699, N40467, N26015, N6199...",N46162,36,-0.663559,N4296,3.456089,-4.119648,"[N4296, N60830, N20841, N35543, N23874, N21666...",...,0,True,autos,finance,True,0.191906,0.075100,0.249918,1.0,0
3,3,6,"[N33276, N2186, N250, N34323, N15476, N36424]",N62365,6,1.750307,N58498,2.158676,-0.408370,"[N58498, N35735, N44283, N45255, N620, N62365,...",...,0,True,foodanddrink,autos,True,0.556340,0.214838,0.689280,1.0,0
4,4,28,"[N24073, N6974, N29911, N63019, N5391, N54496,...",N56969,2,3.166877,N16536,3.173571,-0.006694,"[N16536, N56969, N12075, N17840, N27264, N1456...",...,0,True,lifestyle,news,True,0.690350,0.291341,0.499910,1.0,0


### 10.1 So sánh mẫu đúng và sai theo từng yếu tố

In [7]:
numeric_factors = [
    "history_length",
    "target_train_frequency",
    "target_max_history_cosine",
    "target_mean_history_cosine",
    "hardest_negative_target_cosine",
    "duplicate_negative_count",
    "positive_rank",
]
comparison = analysis_df.groupby("hit_at_k")[numeric_factors].mean().T
comparison.columns = ["sai_top_k", "đúng_top_k"]
comparison["chênh_lệch_sai_trừ_đúng"] = comparison["sai_top_k"] - comparison["đúng_top_k"]

display(comparison)
print("Tỷ lệ target chưa từng xuất hiện làm target trong train:")
display(analysis_df.groupby("hit_at_k")["target_unseen_in_train"].mean().rename(index={False: "sai_top_k", True: "đúng_top_k"}))
print("Tỷ lệ target khác chủ đề chiếm ưu thế trong history:")
display(analysis_df.groupby("hit_at_k")["category_shift"].mean().rename(index={False: "sai_top_k", True: "đúng_top_k"}))
print("Tỷ lệ mẫu có negative bị trùng:")
display(analysis_df.assign(has_duplicate=analysis_df["duplicate_negative_count"] > 0).groupby("hit_at_k")["has_duplicate"].mean().rename(index={False: "sai_top_k", True: "đúng_top_k"}))

,sai_top_k,đúng_top_k,chênh_lệch_sai_trừ_đúng
history_length,39.689519,40.833767,-1.144248
target_train_frequency,10.320938,26.409057,-16.088119
target_max_history_cosine,0.346530,0.411834,-0.065304
target_mean_history_cosine,0.116183,0.154583,-0.038400
hardest_negative_target_cosine,0.440400,0.405266,0.035134
duplicate_negative_count,0.077995,0.079976,-0.001982
positive_rank,38.004285,4.580443,33.423842


Tỷ lệ target chưa từng xuất hiện làm target trong train:


hit_at_k
sai_top_k     0.789776
đúng_top_k    0.682106
Name: target_unseen_in_train, dtype: float64

Tỷ lệ target khác chủ đề chiếm ưu thế trong history:


hit_at_k
sai_top_k     0.702283
đúng_top_k    0.700260
Name: category_shift, dtype: float64

Tỷ lệ mẫu có negative bị trùng:


hit_at_k
sai_top_k     0.074639
đúng_top_k    0.077523
Name: has_duplicate, dtype: float64

### 10.2 Tỷ lệ lỗi theo nhóm

Nếu tỷ lệ lỗi tăng rõ rệt ở một nhóm, yếu tố đó là một giả thuyết nguyên nhân đáng kiểm tra thêm bằng thí nghiệm đối chứng.

In [8]:
diagnostic_df = analysis_df.copy()
diagnostic_df["history_group"] = pd.cut(
    diagnostic_df["history_length"],
    bins=[0, 5, 10, 20, 50, np.inf],
    labels=["1–5", "6–10", "11–20", "21–50", ">50"],
)
diagnostic_df["target_frequency_group"] = pd.cut(
    diagnostic_df["target_train_frequency"],
    bins=[-1, 0, 1, 5, 20, np.inf],
    labels=["unseen", "1", "2–5", "6–20", ">20"],
)
diagnostic_df["history_similarity_group"] = pd.qcut(
    diagnostic_df["target_max_history_cosine"], 4,
    labels=["Q1 thấp", "Q2", "Q3", "Q4 cao"], duplicates="drop",
)
diagnostic_df["hard_negative_group"] = pd.qcut(
    diagnostic_df["hardest_negative_target_cosine"], 4,
    labels=["Q1 thấp", "Q2", "Q3", "Q4 cao"], duplicates="drop",
)

def error_rate_table(column):
    return diagnostic_df.groupby(column, observed=True).agg(
        số_mẫu=("sample_index", "size"),
        tỷ_lệ_sai=("hit_at_k", lambda values: 1 - values.mean()),
        rank_trung_bình=("positive_rank", "mean"),
    )

print("Theo độ dài history")
display(error_rate_table("history_group"))
print("Theo tần suất target trong train")
display(error_rate_table("target_frequency_group"))
print("Theo độ tương đồng target–history")
display(error_rate_table("history_similarity_group"))
print("Theo độ tương đồng giữa target và negative khó nhất")
display(error_rate_table("hard_negative_group"))

Theo độ dài history


,số_mẫu,tỷ_lệ_sai,rank_trung_bình
history_group,,,
1–5,6844,0.667008,27.319550
6–10,7443,0.644901,26.610103
11–20,9941,0.620964,25.364048
21–50,15709,0.595137,24.481253
>50,13818,0.614199,24.602330


Theo tần suất target trong train


,số_mẫu,tỷ_lệ_sai,rank_trung_bình
target_frequency_group,,,
unseen,40260,0.654694,27.336041
1,4106,0.725037,30.089868
2–5,2517,0.351212,13.759634
6–20,1231,0.415922,14.145410
>20,5641,0.468534,15.168410


Theo độ tương đồng target–history


,số_mẫu,tỷ_lệ_sai,rank_trung_bình
history_similarity_group,,,
Q1 thấp,13439,0.729072,29.256567
Q2,13439,0.662698,28.072327
Q3,13438,0.629632,26.923203
Q4 cao,13439,0.462014,17.075080


Theo độ tương đồng giữa target và negative khó nhất


,số_mẫu,tỷ_lệ_sai,rank_trung_bình
hard_negative_group,,,
Q1 thấp,13439,0.464469,15.397351
Q2,13439,0.614480,25.814867
Q3,13438,0.724736,33.342834
Q4 cao,13439,0.679738,26.772602


### 10.3 Gán nhãn nguyên nhân khả dĩ cho từng lỗi

In [9]:
history_similarity_q25 = analysis_df["target_max_history_cosine"].quantile(0.25)
hard_negative_q75 = analysis_df["hardest_negative_target_cosine"].quantile(0.75)
rare_frequency_threshold = 5

def infer_error_reasons(row):
    reasons = []
    if row["history_length"] <= 5:
        reasons.append("history rất ngắn")
    if row["target_unseen_in_train"]:
        reasons.append("target chưa xuất hiện làm target trong train")
    elif row["target_train_frequency"] <= rare_frequency_threshold:
        reasons.append("target hiếm trong train")
    if row["category_shift"]:
        reasons.append("target lệch chủ đề chính của history")
    if row["target_max_history_cosine"] <= history_similarity_q25:
        reasons.append("target ít giống nội dung history")
    if row["hardest_negative_target_cosine"] >= hard_negative_q75:
        reasons.append("có hard negative rất giống target")
    if row["duplicate_negative_count"] > 0:
        reasons.append("negative sampling bị trùng item")
    return reasons or ["không có tín hiệu đơn lẻ rõ ràng"]

error_analysis_df = analysis_df.loc[~analysis_df["hit_at_k"]].copy()
error_analysis_df["possible_reasons"] = error_analysis_df.apply(infer_error_reasons, axis=1)
error_analysis_df["reason_text"] = error_analysis_df["possible_reasons"].str.join("; ")

reason_counts = (
    error_analysis_df["possible_reasons"]
    .explode()
    .value_counts()
    .rename_axis("nguyên_nhân_khả_dĩ")
    .to_frame("số_lỗi")
)
reason_counts["tỷ_lệ_trên_tổng_lỗi"] = reason_counts["số_lỗi"] / len(error_analysis_df)
display(reason_counts)

error_columns = [
    "sample_index", "target_id", "positive_rank", "history_length",
    "target_train_frequency", "target_category", "history_dominant_category",
    "target_max_history_cosine", "hardest_negative_target_cosine",
    "duplicate_negative_count", "score_margin", "reason_text",
]
display(
    error_analysis_df
    .sort_values(["positive_rank", "score_margin"])
    [error_columns]
    .head(20)
)

,số_lỗi,tỷ_lệ_trên_tổng_lỗi
nguyên_nhân_khả_dĩ,,
target chưa xuất hiện làm target trong train,26358,0.789776
target lệch chủ đề chính của history,23438,0.702283
target ít giống nội dung history,9798,0.293582
có hard negative rất giống target,9135,0.273716
history rất ngắn,4565,0.136783
target hiếm trong train,3861,0.115689
negative sampling bị trùng item,2491,0.074639
không có tín hiệu đơn lẻ rõ ràng,231,0.006922


KeyError: 'score_margin'